# Sentiment Analysis Using Word2Vec Embeddings and Support Vector Machine (SVM)

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

This project extends an earlier phase of the same experimentation lab, which used frequency-based representations (Bag-of-Words and TF-IDF) to establish classical baselines. In this phase, I move to **dense, distributed word representations** using **Word2Vec**, and investigate how context-aware embeddings compare to frequency-based features for sentiment classification.

The primary objective is not only to achieve strong predictive performance, but to systematically examine how embedding hyperparameters, sentence-vector aggregation strategy, and preprocessing choices interact — and to understand *why* certain configurations outperform others, rather than just reporting which one wins.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:
* Positive
* Negative

The same train/test split used in the BoW/TF-IDF phase is reused here, with Word2Vec trained only on the training partition to avoid data leakage.

---

## Why Word2Vec, and Why the Preprocessing Changes

Frequency-based methods like BoW and TF-IDF treat words as independent, count-based features — they have no notion of word meaning or context. Word2Vec instead learns dense vector representations from the **local context** each word appears in, capturing semantic relationships (e.g., "great" and "wonderful" ending up close in vector space).

This shift in representation motivates a deliberate shift in preprocessing:

* **Stopwords are retained** (rather than removed, as in the TF-IDF phase), since function words supply the local context Word2Vec learns from — removing them shrinks the effective context window and can degrade embedding quality.
* **No lemmatization** is applied, to preserve distinctions between word forms (e.g., "acting" vs "act") that may carry different contextual signal.
* **Negation handling** carries over from the earlier phase's preprocessing discipline, since negation is critical to sentiment polarity.

---

## Project Pipeline

### 1. Text Preprocessing
* Contraction expansion
* Lowercasing
* Possessive stripping and regex-based cleaning
* Stopwords retained (no removal), no lemmatization
* Corpus construction as tokenized word lists

### 2. Embedding Training
* Word2Vec (Gensim) trained on the training corpus only
* Skip-gram vs. CBOW comparison
* Hyperparameter tuning: `vector_size`, `window`, `epochs`, `min_count`
* Qualitative validation via nearest-neighbor inspection (e.g., words most similar to "great")

### 3. Feature Construction (Sentence Vectors)
* Unweighted mean-pooling of word vectors per review
* TF-IDF-weighted mean-pooling, as a comparison strategy
* Out-of-vocabulary and empty-token edge-case handling

### 4. Model Training
Classical classifiers suited to dense, continuous-valued features are evaluated, including:
* Logistic Regression
* KNearest Neighbors (KNN)
* Support Vector Machines (SVM)
* Random Forest
* XGBoost
* Artificial Neural Network (ANN)
* K-Nearest Neighbors
* Gaussian Naive Bayes

(Multinomial and Bernoulli Naive Bayes are excluded in this phase, as they assume non-negative, count-like input — an assumption Word2Vec's dense, signed vectors violate.)

### 5. Model Evaluation
Performance is assessed using the same metric set as the earlier phase, for direct comparability:
* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score (computed from predicted probabilities, not hard labels)
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single embedding and calling it final, this notebook follows the same experimentation-driven methodology as the BoW/TF-IDF phase. Word2Vec's hyperparameters were tuned incrementally — one dimension at a time — across:

* `sg` (skip-gram vs. CBOW)
* `window` size
* `vector_size`
* `epochs`

with each configuration evaluated via a fixed downstream classifier (KNearest Neighbors (KNN)) to isolate the effect of the embedding itself before introducing classifier choice as a second variable. The best-performing configuration was then carried forward for the full classifier comparison and for the unweighted-vs-TF-IDF-weighted aggregation comparison.

---

## Key Learning Objectives

Through this phase, I aim to:
* Understand how context-based embeddings differ from frequency-based representations, both conceptually and in preprocessing requirements.
* Empirically evaluate how embedding hyperparameters (context window, training epochs, training algorithm) affect downstream classification quality.
* Test the assumption that frequency-based reweighting (TF-IDF) improves embedding-based sentence vectors, and report the result honestly whether or not it confirms the initial hypothesis.
* Compare Word2Vec-based classical models against the BoW/TF-IDF baselines established earlier in the project.
* Build toward sequence-aware models (BiLSTM) that consume these same embeddings, as the next stage beyond static sentence-vector pooling.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)

# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [ ]:
dataset = pd.read_csv('MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 5.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.3 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
import re
import contractions

corpus = []
for i in range(0, len(dataset)):
    review = dataset['review'].iloc[i]
    # Fix contractions (don't -> do not)
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    # remove possessive 's before stripping other punctuation
    review = re.sub(r"'s\b", '', review)
    # Keep only letters and spaces
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split into tokens
    words = review.split()
    # NO stopword removal (keep context), NO lemmatization (keep word forms distinct)
    tokens = words
    corpus.append(tokens)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Encoding Categorical data Using Label Encoding**

In [10]:
from sklearn.preprocessing import LabelEncoder
y = dataset.iloc[:, 0].values
le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
print(y)

[1 1 0 ... 0 0 1]


# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [12]:
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Splitting the dataset into the Training set and Test set**

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size = 0.20, random_state = 0)

# **Creating the Word2Vec model**

In [14]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=X_train,
    vector_size=300,
    window=10,
    min_count=2,
    sg=1,          # skip-gram, per your established methodology
    epochs=20,
    seed=0,
    workers=1      # set to 1 for reproducibility; multi-worker training is non-deterministic even with a seed
)

## **Sentiment Similarity Test:** Iterates through sample sentiment words, checks for vocabulary presence, and prints the top 5 most similar words based on cosine distance using `model.wv.most_similar()`.

In [15]:
test_words = [
    'excellent',
    'wonderful',
    'fantastic',
    'terrible',
    'awful',
    'boring',
    'amazing'
]

for word in test_words:
    if word in model.wv:
        print(f"\nSimilar words to '{word}':")
        for similar_word, score in model.wv.most_similar(word, topn=5):
            print(f"  {similar_word}: {score:.4f}")


Similar words to 'excellent':
  superb: 0.5378
  outstanding: 0.5373
  great: 0.5306
  amazing: 0.5252
  eves: 0.5218

Similar words to 'wonderful':
  great: 0.6062
  excellent: 0.4982
  tink: 0.4808
  amazing: 0.4761
  magnificent: 0.4750

Similar words to 'fantastic':
  great: 0.5182
  brilliant: 0.4868
  excellent: 0.4767
  incredible: 0.4626
  wonderful: 0.4613

Similar words to 'terrible':
  bad: 0.6127
  awful: 0.6037
  horrible: 0.5528
  innane: 0.5456
  unbelieveable: 0.5279

Similar words to 'awful':
  terrible: 0.6037
  bad: 0.5606
  horrible: 0.5206
  missable: 0.5059
  hokie: 0.4934

Similar words to 'boring':
  unsubstantial: 0.5699
  dull: 0.5512
  tarintino: 0.5479
  pointless: 0.5306
  feedings: 0.5219

Similar words to 'amazing':
  excellent: 0.5252
  incredible: 0.5244
  wonderful: 0.4761
  awesome: 0.4672
  outstanding: 0.4656


## Mean Word Vector Aggregation (Dense Feature Extraction)

Translates variable-length token sequences into fixed-dimensional document embeddings for traditional machine learning models.

* **Vocabulary Lookup:** Filters input tokens against `model.wv` to retrieve corresponding Word2Vec embeddings.
* **Centroid Calculation:** Computes the element-wise mean (`axis=0`) over valid word vectors, producing a unigram centroid representation per text sample.
* **Zero-Vector Fallback:** Returns a 300-dimensional zero vector for samples containing exclusively out-of-vocabulary (OOV) terms.
* **Feature Matrix Generation:** Transforms train and test splits into 2D NumPy arrays of shape `(n_samples, 300)`.

In [16]:
def get_average_vector(tokens, model, vector_size):
    # Only keep words that are actually in the trained vocabulary
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    
    if len(valid_vectors) == 0:
        # Edge case: review has no words in vocab (rare, but handle it)
        return np.zeros(vector_size)
    
    return np.mean(valid_vectors, axis=0)

vector_size = 300
X_train_w2v = np.array([get_average_vector(tokens, model, vector_size) for tokens in X_train])
X_test_w2v = np.array([get_average_vector(tokens, model, vector_size) for tokens in X_test])

print("X_train shape:", X_train_w2v.shape)

X_train shape: (19923, 300)


## TF-IDF Weighted Document Embeddings (No Data Leakage)

Combines semantic word embeddings with statistical term frequency weights to generate rich document-level feature representations.

* **Leakage-Free Fitting:** Fits `TfidfVectorizer` exclusively on `X_train` strings, extracting Inverse Document Frequency (IDF) weights into `tfidf_weights`.
* **Fallback Strategy:** Maps missing or single-character tokens (filtered by scikit-learn's default token regex) to `1.0`, matching the theoretical minimum IDF baseline ($\ln(1) + 1$).
* **Weighted Aggregation:** Computes `np.average(vectors, axis=0, weights=weights)` to prioritize high-information, domain-specific terms while down-weighting ubiquitous words.
* **Output Matrices:** Transforms train and test token lists into dense 2D feature matrices (`X_train_tfidf_w2v` and `X_test_tfidf_w2v`) of shape `(n_samples, vector_size)`.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on raw string texts (train only, no leakage)
tfidf = TfidfVectorizer()
tfidf.fit([' '.join(tokens) for tokens in X_train])
tfidf_weights = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def get_tfidf_w2v_vector(tokens, model, vector_size):
    valid_tokens = [w for w in tokens if w in model.wv]
    if not valid_tokens:
        return np.zeros(vector_size)
    
    # sklearn's idf = ln((1+n)/(1+df)) + 1, whose theoretical minimum is exactly 1.0
    # (a word appearing in every document). TfidfVectorizer's default token_pattern
    # also drops single-char tokens (e.g. 'a', 'i'), so they won't be in tfidf_weights
    # even though they're valid Word2Vec tokens -> fall back to that minimum weight.
    weights = [tfidf_weights.get(w, 1.0) for w in valid_tokens]
    vectors = [model.wv[w] for w in valid_tokens]
    
    return np.average(vectors, axis=0, weights=weights)

vector_size = 300
X_train_tfidf_w2v = np.array([get_tfidf_w2v_vector(tokens, model, vector_size) for tokens in X_train])
X_test_tfidf_w2v  = np.array([get_tfidf_w2v_vector(tokens, model, vector_size) for tokens in X_test])

print("X_train shape:", X_train_tfidf_w2v.shape)

X_train shape: (19923, 300)


# **Building LinearSVC Classification Model**

## Training the Building LinearSVC Classification model on the Training set

In [17]:
from sklearn.svm import LinearSVC
classifier = LinearSVC(random_state=0) 
classifier.fit(X_train_w2v, y_train) 

LinearSVC(random_state=0)

# **Predicting the Test set results**

In [18]:
y_pred = classifier.predict(X_test_w2v)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


In [19]:
y_score = classifier.decision_function(X_test_w2v)
print(np.concatenate((y_score.reshape(len(y_score),1), y_test.reshape(len(y_test),1)),1))

[[ 0.59780965  1.        ]
 [ 0.34142546  1.        ]
 [-1.03757098  0.        ]
 ...
 [-0.95810438  0.        ]
 [-1.00214398  0.        ]
 [-0.72109993  0.        ]]


# **Evaluating the Model Performance**

In [20]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score


print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_score))


accuracies = cross_val_score(estimator=classifier, X=X_train_w2v, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[2218  302]
 [ 271 2190]]

Accuracy Score:
0.884962858863682

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.88      0.89      2520
           1       0.88      0.89      0.88      2461

    accuracy                           0.88      4981
   macro avg       0.88      0.89      0.88      4981
weighted avg       0.89      0.88      0.88      4981


ROC-AUC Score:
0.9516980773075857

Mean Accuracy:
0.881845103649049

Standard Deviation:
0.002526670270271206


# Word2Vec Performance Analysis for Text Classification (LinearSVC)

# 1. Objective

The objective of this experiment was to evaluate **Word2Vec embeddings** as a feature representation for **LinearSVC**, the third classifier tested against this embedding family (after Logistic Regression and KNN), completing a set that spans a coefficient-based linear model, a purely distance-based model, and a margin-based linear model.

The experiment proceeded across seven phases, each isolating a different variable: pooling method and scaling (Phase 1), skip-gram vs. CBOW (Phase 2), vector size (Phase 3), window size (Phase 4), vocabulary filtering via `min_count` (Phase 5), training epochs (Phase 6), and regularization strength (Phase 7).

The experiment aims to answer the following research questions:

* Does skip-gram's advantage over CBOW (confirmed for both Logistic Regression and KNN) hold for LinearSVC too?
* Does TF-IDF-weighted pooling help or hurt LinearSVC — and does the answer depend on the base embedding algorithm?
* Does feature scaling matter for a margin-based classifier the way it did for the distance-based KNN?
* How does the best Word2Vec LinearSVC result compare to LinearSVC's BoW and TF-IDF results, and to Word2Vec's other two classifiers?

---

# 2. Experimental Setup

## Dataset

* Final test set: **4,981 documents** (2,520 negative / 2,461 positive).

## Phases

| Phase | What Changed |
| ----- | ------------- |
| 1 | Scaling and pooling method (mean vs. TF-IDF-weighted), base embedding = CBOW, 100d |
| 2 | Skip-gram instead of CBOW, repeating both pooling methods |
| 3 | Vector size 100→200→300 (mean pooling), plus one TF-IDF pooling check at 300d |
| 4 | Window size 5→10 (300d, skip-gram) |
| 5 | `min_count` 2→5 (300d, window=10, skip-gram) |
| 6 | Training epochs 5→10→20 (300d, window=10, `min_count=2`, skip-gram) |
| 7 | Regularization: `LinearSVC(C=0.5)` vs. the default `C=1.0`, on the best Phase 6 embedding |

Every phase includes a qualitative nearest-neighbor check ("excellent", "wonderful") confirming reasonable embedding quality throughout, though neighbor similarity scores drop noticeably as window size and epochs increase (e.g. "excellent" → "outstanding" falls from 0.90 at window=5/epochs=5 to 0.54 at window=10/epochs=20) — expected, since a wider window and more training spreads a word's "signal" across more context words rather than concentrating it on the tightest synonyms.

No automated hyperparameter search was used — every configuration was manually specified and evaluated exactly as configured.

---

# 3. Experimental Results

| # | Phase | Configuration | Accuracy | ROC-AUC | CV Mean | CV Std |
| - | ----- | -------------- | -------- | ------- | ------- | ------ |
| 1 | 1 | CBOW, 100d, mean pool, unscaled | 83.32% | 0.9110 | 83.20% | 0.0015 |
| 2 | 1 | CBOW, 100d, mean pool, scaled | 83.40% | 0.9109 | 83.21% | 0.0019 |
| 3 | 1 | CBOW, 100d, TF-IDF pool, unscaled | 84.84% | 0.9174 | 84.26% | 0.0003 |
| 4 | 1 | CBOW, 100d, TF-IDF pool, scaled | 84.84% | 0.9173 | 84.21% | 0.0007 |
| 5 | 2 | **Skip-gram**, 100d, mean pool | 86.17% | 0.9337 | 86.07% | 0.0024 |
| 6 | 2 | Skip-gram, 100d, TF-IDF pool | 85.63% | 0.9288 | 85.92% | 0.0019 |
| 7 | 3 | Skip-gram, **200d**, mean pool | 86.55% | 0.9396 | 86.76% | 0.0008 |
| 8 | 3 | Skip-gram, **300d**, mean pool | 86.71% | 0.9409 | 87.00% | 0.0011 |
| 9 | 3 | Skip-gram, 300d, TF-IDF pool | 86.45% | 0.9370 | 86.63% | 0.0011 |
| 10 | 4 | Skip-gram, 300d, **window=10**, mean pool | 88.22% | 0.9498 | 87.89% | 0.0014 |
| 11 | 5 | Skip-gram, 300d, window=10, **min_count=5** | 87.81% | 0.9493 | **88.07%** | 0.0012 |
| 12 | 6 | Skip-gram, 300d, window=10, min_count=2, **epochs=10** | 88.13% | 0.9509 | **88.27%** | 0.0020 |
| **13** | 6 | Skip-gram, 300d, window=10, min_count=2, **epochs=20** | **88.50%** | **0.9517** | 88.18% | 0.0025 |
| 14 | 7 | Same as #13, **LinearSVC(C=0.5)** | 88.28% | 0.9518 | 88.17% | 0.0019 |

---

# 4. Word2Vec Experiment Configuration & Results

The following table records every experiment performed in the Word2Vec phase. The Word2Vec model was trained on the training set only, and the resulting word embeddings were converted into document-level representations using either mean pooling or TF-IDF-weighted mean pooling. Unless explicitly stated otherwise, the classifier was `LinearSVC(random_state=0)`.

| Phase | Run | Vector Size | Window | Min Count | SG | Epochs | Representation | Scaling | LinearSVC C | Accuracy | ROC-AUC | Mean Accuracy | Std. Dev. |
| ----- | --: | ----------: | -----: | --------: | -: | -----: | -------------- | ------- | ----------: | -------: | ------: | ------------: | --------: |
| 1     |   1 |         100 |      5 |         2 |  0 |      5 | Mean W2V       | No      |         1.0 |  0.83317 | 0.91097 |       0.83195 |   0.00149 |
| 1     |   2 |         100 |      5 |         2 |  0 |      5 | Mean W2V       | Yes     |         1.0 |  0.83397 | 0.91086 |       0.83205 |   0.00187 |
| 1     |   3 |         100 |      5 |         2 |  0 |      5 | TF-IDF W2V     | No      |         1.0 |  0.84842 | 0.91739 |       0.84264 |   0.00033 |
| 1     |   4 |         100 |      5 |         2 |  0 |      5 | TF-IDF W2V     | Yes     |         1.0 |  0.84842 | 0.91734 |       0.84214 |   0.00068 |
| 2     |   1 |         100 |      5 |         2 |  1 |      5 | Mean W2V       | No      |         1.0 |  0.86167 | 0.93365 |       0.86071 |   0.00245 |
| 2     |   2 |         100 |      5 |         2 |  1 |      5 | TF-IDF W2V     | No      |         1.0 |  0.85625 | 0.92883 |       0.85921 |   0.00186 |
| 3     |   1 |         200 |      5 |         2 |  1 |      5 | Mean W2V       | No      |         1.0 |  0.86549 | 0.93957 |       0.86759 |   0.00082 |
| 3     |   2 |         300 |      5 |         2 |  1 |      5 | Mean W2V       | No      |         1.0 |  0.86709 | 0.94087 |       0.87005 |   0.00107 |
| 3     |   3 |         300 |      5 |         2 |  1 |      5 | TF-IDF W2V     | No      |         1.0 |  0.86449 | 0.93704 |       0.86629 |   0.00109 |
| 4     |   1 |         300 |     10 |         2 |  1 |      5 | Mean W2V       | No      |         1.0 |  0.88215 | 0.94983 |       0.87888 |   0.00141 |
| 5     |   1 |         300 |     10 |         5 |  1 |      5 | Mean W2V       | No      |         1.0 |  0.87814 | 0.94928 |       0.88074 |   0.00119 |
| 6     |   1 |         300 |     10 |         2 |  1 |     10 | Mean W2V       | No      |         1.0 |  0.88135 | 0.95085 |       0.88270 |   0.00198 |
| 6     |   2 |         300 |     10 |         2 |  1 |     20 | Mean W2V       | No      |         1.0 |  0.88496 | 0.95170 |       0.88185 |   0.00253 |
| 7     |   1 |         300 |     10 |         2 |  1 |     20 | Mean W2V       | No      |     **0.5** |  0.88275 | 0.95176 |       0.88174 |   0.00188 |

## Configuration Notes
* `sg=0` → CBOW.
* `sg=1` → Skip-gram.
* Vector size controls the dimensionality of each word embedding.
* Window controls the maximum context distance used during Word2Vec training.
* Min count removes words appearing fewer times than the specified threshold.
* Epochs controls how many times Word2Vec trains over the corpus.
* Mean W2V represents each review by averaging its word vectors.
* TF-IDF W2V represents each review using a TF-IDF-weighted average of its word vectors.
* Scaling means `StandardScaler` was fitted on the training vectors and then applied to the test vectors.
* LinearSVC C controls the regularization strength; the default value used here is `C=1.0`.

## Important reproducibility detail

The **Word2Vec configuration alone is not sufficient to reproduce a run**. Each experiment also depends on the downstream representation and classifier configuration.

For example:
```
Phase 6 — Run 2

Word2Vec
├── vector_size = 300
├── window = 10
├── min_count = 2
├── sg = 1
├── epochs = 20
├── seed = 0
└── workers = 1

Document Representation
└── Mean Word Vector

Scaling
└── None

Classifier
├── LinearSVC
├── C = 1.0
└── random_state = 0
```

---

# 5. Performance Analysis

## Effect of Feature Scaling — Negligible for LinearSVC (Runs 1 vs. 2, and 3 vs. 4)

| Base Config | Unscaled | Scaled | Change |
| ------------ | --------- | ------- | ------ |
| Mean pooling | 83.32% | 83.40% | +0.08 |
| TF-IDF pooling | 84.84% | 84.84% | 0.00 |

Unlike KNN, where scaling gave a clean +0.90 point gain, **LinearSVC is essentially indifferent to feature scaling** on Word2Vec vectors. This makes sense: LinearSVC's margin optimization already involves fitting per-feature coefficients that can absorb differences in feature scale, whereas KNN's raw distance calculation has no such mechanism.

## Effect of Skip-gram vs. CBOW (Runs 1 vs. 5, Isolated)

| Algorithm | Accuracy |
| ---------- | -------- |
| CBOW | 83.32% |
| **Skip-gram** | **86.17%** |

A **+2.85 point** gain from skip-gram — now confirmed as a robust, classifier-agnostic finding across all three Word2Vec experiments in this series (Logistic Regression: +2.77, KNN: +4.05, LinearSVC: +2.85).

## Effect of Pooling Method — Depends on the Base Embedding Algorithm

*(This is the most important finding in this experiment.)*

| Base Embedding | Mean Pooling | TF-IDF Pooling | Change |
| ---------------- | -------------- | ----------------- | ------ |
| CBOW, 100d | 83.32% | 84.84% | **+1.52** |
| Skip-gram, 100d | 86.17% | 85.63% | **-0.54** |
| Skip-gram, 300d | 86.71% | 86.45% | **-0.26** |

TF-IDF-weighted pooling **helps** when the base embedding is CBOW, but **hurts** when the base embedding is skip-gram — confirmed twice at different vector sizes for the skip-gram case. This is a more nuanced result than either the Logistic Regression report (where TF-IDF pooling always hurt) or the KNN report (where it always helped) found on their own — for LinearSVC specifically, the answer depends on a *second* variable, not just the classifier. A plausible explanation: CBOW embeddings are known to be somewhat "blurrier" representations of a word's context (they predict a word from its surrounding context rather than the reverse), so TF-IDF's emphasis on rare/distinctive words may compensate for that blurriness — while skip-gram embeddings are already sharper, and additional reweighting on top of them mostly just injects noise from poorly-estimated rare-word vectors, similar to the mechanism identified in the Logistic Regression report.

## Effect of Vector Size (Runs 5, 7, 8, Isolated)

| Vector Size | Accuracy |
| ------------- | -------- |
| 100 | 86.17% |
| 200 | 86.55% |
| 300 | 86.71% |

A small, monotonic gain with diminishing returns (+0.38 then +0.16) — matching the pattern (and even the approximate magnitude) found for Logistic Regression, not the flat-to-negative pattern found for KNN. This reinforces the KNN report's conclusion that dimensionality sensitivity is specific to distance-based methods — both linear classifiers tested (Logistic Regression, LinearSVC) benefit similarly from more dimensions.

## Effect of Window Size (Run 8 vs. 10, Isolated)

| Window | Accuracy |
| ------- | -------- |
| 5 | 86.71% |
| **10** | **88.22%** |

The single largest lever in this experiment: **+1.51 points** from doubling the context window — consistent with both earlier Word2Vec reports finding wider windows help.

## Effect of `min_count` (Run 10 vs. 11)

| min_count | Test Accuracy | CV Mean |
| ---------- | --------------- | ------- |
| 2 | **88.22%** | 87.89% |
| 5 | 87.81% | **88.07%** |

Filtering out more rare words (`min_count=5`) slightly *hurt* single-split test accuracy but slightly *improved* the CV mean — another case in this project where the two metrics disagree on which configuration is better. The likely explanation: dropping rare words removes some noisy, poorly-estimated vectors (helping generalization, reflected in CV mean) but also removes some genuinely useful rare-word signal (hurting the specific test split).

## Effect of Training Epochs (Runs 10, 12, 13)

| Epochs | Test Accuracy | CV Mean |
| ------- | --------------- | ------- |
| 5 | 88.22% | 87.89% |
| 10 | 88.13% | **88.27%** |
| **20** | **88.50%** | 88.18% |

Test accuracy dips slightly at epochs=10 before recovering and exceeding its epochs=5 level at epochs=20, while CV mean peaks at epochs=10 and is essentially flat into epochs=20. The two metrics don't agree on a single "best" epoch count here — see Section 7 for how this affects the best-model choice.

## Effect of Regularization (Run 13 vs. 14)

| C | Test Accuracy | CV Mean |
| - | --------------- | ------- |
| 1.0 (default) | **88.50%** | 88.18% |
| 0.5 | 88.28% | 88.17% |

Unlike the TF-IDF LinearSVC experiment (where lowering `C` to 0.5 was essential to reaching the best result), regularization tuning **did not help** here — the default `C=1.0` was already effectively optimal for this embedding. A useful reminder that a hyperparameter finding from one feature representation doesn't automatically transfer to another, even for the same classifier.

---

# 6. Precision and Recall Analysis

### Run 5 (Skip-gram, Mean Pooling, 100d)

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0 | 0.87 | 0.85 | 0.86 |
| 1 | 0.85 | 0.87 | 0.86 |

### Best Model — Run 13

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0 | 0.89 | 0.88 | 0.89 |
| 1 | 0.88 | 0.89 | 0.88 |

Class balance stays tight and near-symmetric across the whole experiment — consistent with the other two Word2Vec reports.

---

# 7. Cross-Validation Analysis

| Run | Configuration | Test Accuracy | CV Mean | CV Std |
| --- | -------------- | --------------- | ------- | ------ |
| 11 | min_count=5 | 87.81% | 88.07% | 0.0012 |
| 12 | epochs=10 | 88.13% | **88.27%** | 0.0020 |
| **13** | **epochs=20 (best by test accuracy)** | **88.50%** | 88.18% | 0.0025 |

Run 12 actually has the highest CV mean of the entire experiment (88.27%), narrowly ahead of Run 13's 88.18% — even though Run 13 wins on single-split test accuracy. Both are very close, and either would be a reasonable choice; Run 13 is reported as the headline "best model" since it's the highest test accuracy, but Run 12 (fewer training epochs, marginally better CV mean) is a defensible, slightly cheaper alternative.

---

# 8. Best Model Configuration

```python
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=X_train, vector_size=300, window=10,
    min_count=2, sg=1, epochs=20, seed=0, workers=1,
)

def get_average_vector(tokens, model, vector_size):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(valid_vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(valid_vectors, axis=0)
vector_size = 300
X_train_w2v = np.array([get_average_vector(tokens, model, vector_size) for tokens in X_train])
X_test_w2v = np.array([get_average_vector(tokens, model, vector_size) for tokens in X_test])
print("X_train shape:", X_train_w2v.shape)

from sklearn.svm import LinearSVC
classifier = LinearSVC(random_state=0)
classifier.fit(X_train_w2v, y_train)
```

Performance:

* Accuracy = **88.50%**
* Precision (Class 0 / 1) = **0.89 / 0.88**
* Recall (Class 0 / 1) = **0.88 / 0.89**
* ROC-AUC = **0.9517**
* Cross-Validation Accuracy = **88.18%**
* Cross-Validation Standard Deviation = **0.0025**

*(As noted in Section 6, Run 12 — epochs=10 — has a marginally higher CV mean and is a reasonable alternative if training cost matters.)*

---

# 9. Comparison with LinearSVC on BoW and TF-IDF

| Representation | Best Config | Accuracy | ROC-AUC |
| ---------------- | ------------ | -------- | ------- |
| BoW | 30K features | 87.51% | 0.8753 |
| **TF-IDF** | **30K features, C=0.5** | **90.18%** | 0.9020 |
| Word2Vec | 300d, window=10, epochs=20, mean pool | 88.50% | 0.9517 |

On accuracy, Word2Vec lands between BoW and TF-IDF for LinearSVC — better than raw counts, but still behind TF-IDF's weighting. TF-IDF's built-in down-weighting of common terms appears to give LinearSVC a stronger signal than either the raw counts of BoW or the averaged embeddings of Word2Vec.

---

# 10. Synthesis: Word2Vec Across Three Classifiers


| Finding | Logistic Regression | KNN | LinearSVC |
| -------- | ---------------------- | ---- | ---------- |
| Skip-gram beats CBOW | +2.77 pts | +4.05 pts | +2.85 pts |
| More vector dimensions help | Yes (diminishing) | Flat/negative | Yes (diminishing) |
| TF-IDF-weighted pooling helps | Always hurts | Always helps | **Depends on CBOW vs. skip-gram** |
| Feature scaling matters | *(not tested)* | Meaningful gain | Negligible |
| Wider window helps | Yes | *(not isolated)* | Yes |
| More training epochs help | Yes | Yes (with caveats) | Mostly, with a dip at epochs=10 |

Two findings are **universal** across every classifier tested: skip-gram consistently beats CBOW, and a wider training window consistently helps. Two findings are **classifier-dependent**: vector-size sensitivity splits along the distance-based vs. coefficient-based line (KNN vs. the two linear models), and TF-IDF pooling's effect is genuinely complicated — helping KNN, hurting Logistic Regression, and for LinearSVC depending on a *third* factor (the base embedding algorithm) that neither other report tested for. This last point is the most important practical takeaway of the whole three-report series: **a pooling or embedding choice validated for one classifier should not be assumed to transfer to another without re-testing.**

---

# 11. Discussion

This experiment reinforces two of the most robust findings from the Word2Vec phase of this project — skip-gram's advantage and the value of a wider context window — while adding a genuinely new layer of nuance to the pooling-method question. Where the Logistic Regression and KNN reports each seemed to support a simple, classifier-level rule ("TF-IDF pooling always hurts" / "always helps"), this experiment shows that rule breaking down further: the same classifier (LinearSVC) gets different answers depending on the embedding algorithm underneath it.

Regularization tuning (`C`), which was essential to LinearSVC's best TF-IDF result, provided no benefit here — a useful reminder that hyperparameter choices are tied to the specific feature representation, not just the classifier.

---

# 12. Final Conclusion

This experiment evaluated Word2Vec embeddings for LinearSVC across seven phases and fourteen configurations. The best model used a **300-dimensional skip-gram embedding (window=10, 20 epochs), mean pooling, and default `LinearSVC(C=1.0)`**, achieving **88.50% accuracy** and **0.9517 ROC-AUC** — ahead of LinearSVC's BoW result (87.51%) but behind its TF-IDF result (90.18%).

The most valuable output of this experiment is the Section 9 synthesis across all three Word2Vec classifiers tested in this project: two findings (skip-gram advantage, window size) generalize cleanly, but the pooling-method question turns out to depend on more than just the classifier — it depends on the embedding algorithm too. Future embedding-configuration decisions in this project should be validated per classifier rather than assumed to transfer.